In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))

import json 
import copy
import os
import torch
import tqdm
import numpy as np
import matplotlib.pyplot as plt
import torch
import numpy as np
from matplotlib import colors
import torchvision
import torch.nn.functional as F
import pandas as pd
from matplotlib.colors import ListedColormap

from src.utils import config_helper
from src.data.get_dataloaders import get_dataloaders
from src.data.dataset import DeforestationForecastingDataset
from src.algorithm.models.get_model import  get_model
from src.algorithm.loss import get_loss_fn


/home/jan/miniconda3/envs/smarties_fresh/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Need a config for data loading the dataset

config_path = '../config/sample_config.json'
config = config_helper(config_path)
config['annotation_file'] = f"../{config['annotation_file']}"
config['data']['deforestation_map'] = f"../{config['data']['deforestation_map']}"
config['inputs'] = ['deforestation_map']
device = 'cuda'

val_dl = get_dataloaders(['val'], config)['val']


In [4]:
from torchmetrics.functional import jaccard_index, precision, recall, f1_score 
from torchmetrics.functional.classification import binary_precision_recall_curve
from sklearn.metrics import average_precision_score, precision_recall_curve, PrecisionRecallDisplay, roc_auc_score
from torchmetrics.classification import PrecisionRecallCurve
from torchmetrics import MeanAbsoluteError, MeanSquaredError, R2Score, PearsonCorrCoef, MetricCollection

def compute_metrics(targets, preds, dataset_name, regression_metrics=False):

    preds = preds.to(dtype=torch.float16)
    targets = targets.to(dtype=torch.int16)

    mask = targets != -1
    valid_targets = targets[mask].flatten()
    valid_preds = preds[mask].flatten()
    
    
    if regression_metrics:
        binarized_preds = (preds >= 0.5).int() 
        pred_pos = ((binarized_preds == 1) & mask).sum(dim=(1,2)).float()
        targ_pos = ((targets == 1) & mask).sum(dim=(1,2)).float()
        # stack for torchmetrics
        # exclude tiles with no valid pixels
        has_valid_data = mask.sum(dim=(1,2)) > 0
        pred_pos, targ_pos = pred_pos[has_valid_data], targ_pos[has_valid_data]
        pred_pos = pred_pos.unsqueeze(1)
        targ_pos = targ_pos.unsqueeze(1)
        reg_metrics_coll = MetricCollection({
            f'{dataset_name}_MAE': MeanAbsoluteError(),
            f'{dataset_name}_MSE': MeanSquaredError(),
            f'{dataset_name}_RMSE': MeanSquaredError(squared=False),
            f'{dataset_name}_R2': R2Score(),
            f'{dataset_name}_PearsonR': PearsonCorrCoef(),
        })

        reg_metrics = reg_metrics_coll(pred_pos, targ_pos)


    _prec, _rec, _thresh = binary_precision_recall_curve(valid_preds, valid_targets,ignore_index=-1)


    metrics = {
        f'{dataset_name}_iou' : jaccard_index(preds, targets, task='binary', num_classes=1, ignore_index=-1),
        f'{dataset_name}_f1' : f1_score(preds, targets, task='binary', num_classes=1, ignore_index=-1),
        f'{dataset_name}_precision' : precision(preds, targets, task='binary', num_classes=1, ignore_index=-1),
        f'{dataset_name}_recall' : recall(preds, targets, task='binary', num_classes=1, ignore_index=-1),
        f'{dataset_name}_average_precision' : average_precision_score(valid_targets.flatten(), valid_preds.flatten()),
        f'{dataset_name}_ROC_AUC' : roc_auc_score(valid_targets.flatten(), valid_preds.flatten()),
    }

    if regression_metrics:
        metrics.update(reg_metrics)


    return metrics



## Define model

In [5]:
import torch
import numpy as np
from scipy.ndimage import distance_transform_edt

class ExponentialDecayBaseline:
    """
    Risk decays exponentially with distance to nearest deforestation (value=1).
    Output risk only for forest pixels (value=0). Other values left at 0.
    """
    def __init__(self, decay_distance):
        self.decay_distance_pixels = float(decay_distance)

    def predict(self, deforestation_map: torch.Tensor) -> torch.Tensor:
        # deforestation_map: (B,H,W) with {1=deforestation, 0=forest, -1=invalid}
        dm = deforestation_map.cpu().numpy()
        risk_maps = []

        for b in range(dm.shape[0]):
            is_deforestation = (dm[b] == 1)
            is_forest        = (dm[b] == 0)

            if is_deforestation.any():
                # distance to nearest deforestation pixel
                distance_to_defor = distance_transform_edt(~is_deforestation)
            else:
                distance_to_defor = np.full(dm[b].shape, np.inf, dtype=float)

            risk = np.exp(-distance_to_defor / self.decay_distance_pixels)

            # keep risk only on forest pixels
            risk[~is_forest] = 0.0

            risk_maps.append(risk.astype(np.float32))

        return torch.from_numpy(np.stack(risk_maps))


# Tune parameter

In [136]:

for dist in [2.6, 2.8, 3., 3.2, 3.4]:
    model = ExponentialDecayBaseline(decay_distance=dist)


    preds_batches = []
    targets_batches = []
    inputs_batches = []

    for idx, (input_batch, labels_batch) in tqdm.tqdm(enumerate(val_dl)):

        def_map = input_batch['deforestation_map'][:,-1].squeeze()
        y_hat = model.predict(def_map)

        labels_batch = labels_batch.cpu().detach()[:,-1].squeeze()

        preds_batches.append(y_hat)
        targets_batches.append(labels_batch)
        inputs_batches.append(def_map)

    targets = torch.cat(targets_batches, axis=0).cpu().detach()
    preds = torch.cat(preds_batches, axis=0).cpu().detach()
    inputs = torch.cat(inputs_batches, axis=0).cpu().detach()

    metrics = compute_metrics(targets, preds, 'val', regression_metrics=True)
    print(dist)
    print(metrics)

136it [00:20,  6.50it/s]


2.6
{'val_iou': tensor(0.2208), 'val_f1': tensor(0.3618), 'val_precision': tensor(0.3851), 'val_recall': tensor(0.3411), 'val_average_precision': np.float64(0.2608399352036), 'val_ROC_AUC': np.float64(0.8516635842866424), 'val_MAE': tensor(30.3188), 'val_MSE': tensor(5985.6709), 'val_PearsonR': tensor(0.5105), 'val_R2': tensor(0.2550), 'val_RMSE': tensor(77.3671)}


136it [00:20,  6.51it/s]


2.8
{'val_iou': tensor(0.2208), 'val_f1': tensor(0.3618), 'val_precision': tensor(0.3851), 'val_recall': tensor(0.3411), 'val_average_precision': np.float64(0.26084779330019825), 'val_ROC_AUC': np.float64(0.8516697579115049), 'val_MAE': tensor(30.3188), 'val_MSE': tensor(5985.6709), 'val_PearsonR': tensor(0.5105), 'val_R2': tensor(0.2550), 'val_RMSE': tensor(77.3671)}


136it [00:20,  6.52it/s]


3.0
{'val_iou': tensor(0.2227), 'val_f1': tensor(0.3643), 'val_precision': tensor(0.3282), 'val_recall': tensor(0.4094), 'val_average_precision': np.float64(0.260850917791662), 'val_ROC_AUC': np.float64(0.8516623496291589), 'val_MAE': tensor(37.7741), 'val_MSE': tensor(6161.2593), 'val_PearsonR': tensor(0.5122), 'val_R2': tensor(0.2332), 'val_RMSE': tensor(78.4937)}


136it [00:20,  6.51it/s]


3.2
{'val_iou': tensor(0.2227), 'val_f1': tensor(0.3643), 'val_precision': tensor(0.3282), 'val_recall': tensor(0.4094), 'val_average_precision': np.float64(0.26085221103742096), 'val_ROC_AUC': np.float64(0.8516565286480546), 'val_MAE': tensor(37.7741), 'val_MSE': tensor(6161.2593), 'val_PearsonR': tensor(0.5122), 'val_R2': tensor(0.2332), 'val_RMSE': tensor(78.4937)}


136it [00:20,  6.49it/s]


3.4
{'val_iou': tensor(0.2136), 'val_f1': tensor(0.3520), 'val_precision': tensor(0.2805), 'val_recall': tensor(0.4723), 'val_average_precision': np.float64(0.2608527384804993), 'val_ROC_AUC': np.float64(0.8516531415549285), 'val_MAE': tensor(50.6128), 'val_MSE': tensor(7652.8354), 'val_PearsonR': tensor(0.5046), 'val_R2': tensor(0.0475), 'val_RMSE': tensor(87.4805)}


In [ ]:
# Based on the hyperparameter tuning above, decay_distance has been set to 3.0


## Get metrics on test set

In [6]:
# Need a config for loading the dataset


df = pd.read_csv(config['annotation_file'])
df = df.loc[df.split == 'test']
df['split'] = 'val'
df.to_csv("../data/test_annotation.csv", index=False)

config['annotation_file'] = "../data/test_annotation.csv"

device = 'cpu'
config['wandb']['log_to_wandb'] = False



In [7]:
model = ExponentialDecayBaseline(decay_distance=3)

In [21]:
all_metrics = {}

year = 2012
_config = config.copy()
seq_len = _config['val_period']['input_seq_len']
print(year, "-", year+seq_len+1)


year_min, year_max = year, year + seq_len + 1
time_period = f'{year_min}_{year_max}'
_config['val_period']  = {'year_min': year_min, 'year_max': year_max, 'input_seq_len': seq_len}

val_dl = get_dataloaders(['val'], _config)['val']
print("dataset size: ", len(val_dl.dataset))

preds_batches = []
targets_batches = []

for idx, (input_batch, labels_batch) in tqdm.tqdm(enumerate(val_dl)):

    def_map = input_batch['deforestation_map'][:,-1].squeeze()
    y_hat = model.predict(def_map)

    labels_batch = labels_batch.cpu().detach()[:,-1].squeeze()

    preds_batches.append(y_hat)
    targets_batches.append(labels_batch)

try:
    targets = torch.cat(targets_batches, axis=0).cpu().detach()
    preds = torch.cat(preds_batches, axis=0).cpu().detach()
except RuntimeError:
    targets_batches[-1] = targets_batches[-1].unsqueeze(0)
    targets = torch.cat(targets_batches, axis=0).cpu().detach()
    preds_batches[-1] = preds_batches[-1].unsqueeze(0)
    preds = torch.cat(preds_batches, axis=0).cpu().detach()


metrics = compute_metrics(targets, preds, 'val', regression_metrics=True)
print(metrics)
all_metrics[time_period] = metrics

2012 - 2017
dataset size:  5036


79it [00:16,  4.69it/s]


{'val_iou': tensor(0.2250), 'val_f1': tensor(0.3674), 'val_precision': tensor(0.3350), 'val_recall': tensor(0.4067), 'val_average_precision': 0.2624748574820074, 'val_ROC_AUC': 0.825954611270427, 'val_MAE': tensor(42.3274), 'val_MSE': tensor(8776.1699), 'val_PearsonR': tensor(0.4903), 'val_R2': tensor(0.2198), 'val_RMSE': tensor(93.6812)}


## Test metrics:
    
{'val_iou': tensor(0.2250), 'val_f1': tensor(0.3674), 'val_precision': tensor(0.3350), 'val_recall': tensor(0.4067), 'val_average_precision': 0.2624748574820074, 'val_ROC_AUC': 0.825954611270427, 'val_MAE': tensor(42.3274), 'val_MSE': tensor(8776.1699), 'val_PearsonR': tensor(0.4903), 'val_R2': tensor(0.2198), 'val_RMSE': tensor(93.6812)}


# Get metrics for multiple years

In [25]:
FIRST_YEAR,LAST_YEAR = 2012, 2017

all_metrics = {}

for year in range(FIRST_YEAR, LAST_YEAR):
    _config = config.copy()
    seq_len = _config['val_period']['input_seq_len']
    print(year, "-", year+seq_len+1)
    
    
    year_min, year_max = year, year + seq_len + 1
    time_period = f'{year_min}_{year_max}'
    _config['val_period']  = {'year_min': year_min, 'year_max': year_max, 'input_seq_len': seq_len}
    
    val_dl = get_dataloaders(['val'], _config)['val']
    print("dataset size: ", len(val_dl.dataset))

    preds_batches = []
    targets_batches = []

    for idx, (input_batch, labels_batch) in tqdm.tqdm(enumerate(val_dl)):

        def_map = input_batch['deforestation_map'][:,-1].squeeze()
        y_hat = model.predict(def_map)

        labels_batch = labels_batch.cpu().detach()[:,-1].squeeze()

        preds_batches.append(y_hat)
        targets_batches.append(labels_batch)

    try:
        targets = torch.cat(targets_batches, axis=0).cpu().detach()
        preds = torch.cat(preds_batches, axis=0).cpu().detach()
    except RuntimeError:
        targets_batches[-1] = targets_batches[-1].unsqueeze(0)
        targets = torch.cat(targets_batches, axis=0).cpu().detach()
        preds_batches[-1] = preds_batches[-1].unsqueeze(0)
        preds = torch.cat(preds_batches, axis=0).cpu().detach()
        
        
    metrics = compute_metrics(targets, preds, 'val', regression_metrics=True)
    
    metrics = {
        k: (v.item() if hasattr(v, "item") else float(v))
        for k, v in metrics.items()
    }
    print(metrics)
    all_metrics[time_period] = metrics

2012 - 2017
dataset size:  5036


79it [00:36,  2.16it/s]


{'val_iou': 0.22501519322395325, 'val_f1': 0.3673671782016754, 'val_precision': 0.33498960733413696, 'val_recall': 0.4066731333732605, 'val_average_precision': 0.2624748574820074, 'val_ROC_AUC': 0.825954611270427, 'val_MAE': 42.327392578125, 'val_MSE': 8776.169921875, 'val_PearsonR': 0.49033474922180176, 'val_R2': 0.2198479175567627, 'val_RMSE': 93.68121337890625}
2013 - 2018
dataset size:  5361


84it [00:38,  2.20it/s]


{'val_iou': 0.22150252759456635, 'val_f1': 0.36267223954200745, 'val_precision': 0.2994382977485657, 'val_recall': 0.45976272225379944, 'val_average_precision': 0.254993686617377, 'val_ROC_AUC': 0.8671543571763843, 'val_MAE': 42.1481819152832, 'val_MSE': 6189.39892578125, 'val_PearsonR': 0.5477219820022583, 'val_R2': 0.18018567562103271, 'val_RMSE': 78.67273712158203}
2014 - 2019
dataset size:  5446


...

{'val_iou': 0.19341596961021423, 'val_f1': 0.3241384029388428, 'val_precision': 0.3159216046333313, 'val_recall': 0.33279404044151306, 'val_average_precision': 0.2294472781680204, 'val_ROC_AUC': 0.8141291537216591, 'val_MAE': 47.775577545166016, 'val_MSE': 12920.1240234375, 'val_PearsonR': 0.46720772981643677, 'val_R2': 0.21568375825881958, 'val_RMSE': 113.6667251586914}
2016 - 2021
dataset size:  5907


93it [00:40,  2.31it/s]


{'val_iou': 0.16288602352142334, 'val_f1': 0.2801409959793091, 'val_precision': 0.26196759939193726, 'val_recall': 0.30102381110191345, 'val_average_precision': 0.18449367725886842, 'val_ROC_AUC': 0.7978303144939558, 'val_MAE': 50.77255630493164, 'val_MSE': 13572.212890625, 'val_PearsonR': 0.42266398668289185, 'val_R2': 0.16517186164855957, 'val_RMSE': 116.49983978271484}
2017 - 2022
dataset size:  5810


91it [00:40,  2.27it/s]


{'val_iou': 0.1537061184644699, 'val_f1': 0.26645627617836, 'val_precision': 0.2973482310771942, 'val_recall': 0.2413790374994278, 'val_average_precision': 0.1915831740130936, 'val_ROC_AUC': 0.7551010930539082, 'val_MAE': 57.73314666748047, 'val_MSE': 21703.578125, 'val_PearsonR': 0.35818931460380554, 'val_R2': 0.11976134777069092, 'val_RMSE': 147.3213348388672}


In [23]:
all_metrics

{'2012_2017': {'val_iou': tensor(0.2250),
  'val_f1': tensor(0.3674),
  'val_precision': tensor(0.3350),
  'val_recall': tensor(0.4067),
  'val_average_precision': 0.2624748574820074,
  'val_ROC_AUC': 0.825954611270427,
  'val_MAE': tensor(42.3274),
  'val_MSE': tensor(8776.1699),
  'val_PearsonR': tensor(0.4903),
  'val_R2': tensor(0.2198),
  'val_RMSE': tensor(93.6812)},
 '2013_2018': {'val_iou': tensor(0.2215),
  'val_f1': tensor(0.3627),
  'val_precision': tensor(0.2994),
  'val_recall': tensor(0.4598),
  'val_average_precision': 0.254993686617377,
  'val_ROC_AUC': 0.8671543571763843,
  'val_MAE': tensor(42.1482),
  'val_MSE': tensor(6189.3989),
  'val_PearsonR': tensor(0.5477),
  'val_R2': tensor(0.1802),
  'val_RMSE': tensor(78.6727)},
 '2014_2019': {'val_iou': tensor(0.2216),
  'val_f1': tensor(0.3628),
  'val_precision': tensor(0.3210),
  'val_recall': tensor(0.4170),
  'val_average_precision': 0.25891703195916305,
  'val_ROC_AUC': 0.8540158965660222,
  'val_MAE': tensor(40.542

In [ ]:
{'2010_2015': {'val_iou': tensor(0.2096),
  'val_f1': tensor(0.3465),
  'val_precision': tensor(0.2831),
  'val_recall': tensor(0.4464),
  'val_average_precision': np.float64(0.23760131699405454),
  'val_ROC_AUC': np.float64(0.8602350381690842),
  'val_MAE': tensor(36.9054),
  'val_MSE': tensor(4588.),
  'val_PearsonR': tensor(0.4968),
  'val_R2': tensor(0.0794),
  'val_RMSE': tensor(67.7348)},
 '2011_2016': {'val_iou': tensor(0.2152),
  'val_f1': tensor(0.3542),
  'val_precision': tensor(0.3085),
  'val_recall': tensor(0.4157),
  'val_average_precision': np.float64(0.2483242149536933),
  'val_ROC_AUC': np.float64(0.8603893834701163),
  'val_MAE': tensor(33.3143),
  'val_MSE': tensor(4911.0884),
  'val_PearsonR': tensor(0.5286),
  'val_R2': tensor(0.2101),
  'val_RMSE': tensor(70.0792)},
 '2012_2017': {'val_iou': tensor(0.2182),
  'val_f1': tensor(0.3583),
  'val_precision': tensor(0.3231),
  'val_recall': tensor(0.4021),
  'val_average_precision': np.float64(0.25023593428158764),
  'val_ROC_AUC': np.float64(0.8348900344871557),
  'val_MAE': tensor(34.7660),
  'val_MSE': tensor(7069.8457),
  'val_PearsonR': tensor(0.4790),
  'val_R2': tensor(0.1948),
  'val_RMSE': tensor(84.0824)},
 '2013_2018': {'val_iou': tensor(0.2183),
  'val_f1': tensor(0.3583),
  'val_precision': tensor(0.2897),
  'val_recall': tensor(0.4696),
  'val_average_precision': np.float64(0.2503850217488603),
  'val_ROC_AUC': np.float64(0.8893931990773146),
  'val_MAE': tensor(31.5791),
  'val_MSE': tensor(3539.2705),
  'val_PearsonR': tensor(0.5953),
  'val_R2': tensor(0.1324),
  'val_RMSE': tensor(59.4918)},
 '2014_2019': {'val_iou': tensor(0.2132),
  'val_f1': tensor(0.3515),
  'val_precision': tensor(0.3125),
  'val_recall': tensor(0.4017),
  'val_average_precision': np.float64(0.24476199567920592),
  'val_ROC_AUC': np.float64(0.8550827063858424),
  'val_MAE': tensor(30.1743),
  'val_MSE': tensor(5093.2490),
  'val_PearsonR': tensor(0.5232),
  'val_R2': tensor(0.2352),
  'val_RMSE': tensor(71.3670)},
 '2015_2020': {'val_iou': tensor(0.1873),
  'val_f1': tensor(0.3156),
  'val_precision': tensor(0.2961),
  'val_recall': tensor(0.3378),
  'val_average_precision': np.float64(0.21430027412285738),
  'val_ROC_AUC': np.float64(0.8281217814954565),
  'val_MAE': tensor(33.0904),
  'val_MSE': tensor(7075.0698),
  'val_PearsonR': tensor(0.4651),
  'val_R2': tensor(0.1954),
  'val_RMSE': tensor(84.1134)},
 '2016_2021': {'val_iou': tensor(0.1648),
  'val_f1': tensor(0.2829),
  'val_precision': tensor(0.2508),
  'val_recall': tensor(0.3245),
  'val_average_precision': np.float64(0.1796476995357221),
  'val_ROC_AUC': np.float64(0.8324378608389877),
  'val_MAE': tensor(30.7891),
  'val_MSE': tensor(5016.3472),
  'val_PearsonR': tensor(0.4716),
  'val_R2': tensor(0.1559),
  'val_RMSE': tensor(70.8262)},
 '2017_2022': {'val_iou': tensor(0.1568),
  'val_f1': tensor(0.2711),
  'val_precision': tensor(0.2932),
  'val_recall': tensor(0.2521),
  'val_average_precision': np.float64(0.1860210723497109),
  'val_ROC_AUC': np.float64(0.7917400805790256),
  'val_MAE': tensor(33.9134),
  'val_MSE': tensor(8148.3022),
  'val_PearsonR': tensor(0.4147),
  'val_R2': tensor(0.1596),
  'val_RMSE': tensor(90.2679)},
 '2018_2023': {'val_iou': tensor(0.1335),
  'val_f1': tensor(0.2355),
  'val_precision': tensor(0.2165),
  'val_recall': tensor(0.2583),
  'val_average_precision': np.float64(0.14166085349322635),
  'val_ROC_AUC': np.float64(0.7792415517895497),
  'val_MAE': tensor(33.6527),
  'val_MSE': tensor(5735.9839),
  'val_PearsonR': tensor(0.3646),
  'val_R2': tensor(0.0308),
  'val_RMSE': tensor(75.7363)},
 '2019_2024': {'val_iou': tensor(0.1335),
  'val_f1': tensor(0.2356),
  'val_precision': tensor(0.2028),
  'val_recall': tensor(0.2810),
  'val_average_precision': np.float64(0.12935467774938292),
  'val_ROC_AUC': np.float64(0.8055817997171689),
  'val_MAE': tensor(26.0677),
  'val_MSE': tensor(5135.1606),
  'val_PearsonR': tensor(0.3241),
  'val_R2': tensor(0.0527),
  'val_RMSE': tensor(71.6600)}}